In [38]:
import pandas as pd
import numpy as np

In [39]:
cluster1995 = pd.read_csv(r"C:\Users\ayaan\OneDrive - London School of Economics\Academics\MPA_DSPP\Moody's - Capstone Project\Capstone-Ayaan-ASUS\MASTER\clustersagg.csv")
master = pd.read_csv(r"C:\Users\ayaan\OneDrive - London School of Economics\Academics\MPA_DSPP\Moody's - Capstone Project\Capstone-Ayaan-ASUS\MASTER\Master.csv", index_col=0)

In [40]:
include_list = ['AGO', 'ARE', 'AZE', 'BFA', 'BHR', 'BOL', 'CHL', 'CIV', 'CMR',
       'COD', 'COG', 'DZA', 'ECU', 'EGY', 'ETH', 'GAB', 'GHA', 'GIN',
       'GNQ', 'IDN', 'IRN', 'IRQ', 'KAZ', 'KEN', 'KWT', 'LAO', 'LBR',
       'LBY', 'MDG', 'MLI', 'MMR', 'MNG', 'MOZ', 'MWI', 'MYS', 'NER',
       'NGA', 'OMN', 'PNG', 'QAT', 'RUS', 'RWA', 'SAU', 'TCD', 'TGO',
       'TTO', 'TZA', 'UGA', 'UZB', 'VEN', 'VNM', 'YEM', 'ZMB', 'ZWE']

In [41]:
master = master[(master["Country Code"].isin(include_list))]

In [42]:
master = pd.merge(master, cluster1995[["Country Code", "Cluster"]], on="Country Code", how= "left")
master.head()

,Country Code,Country Name,Year,Access to electricity (% of population),Adjusted savings: gross savings (% of GNI),Agriculture,Capital depreciation rate,Clientelism index,"Death rates, crude per 1000 people",Domestic credit to private sector (% of GDP),...,"Use of IMF credit (DOD, current US$)",Total_Production,Total_Reserves,Total_Production_Value,Total_Reserves_Value,Hydrocarbons_Dominant,Subsoil_Metals_Dominant,Precious_Metals_Dominant,Population,Cluster
0,AGO,Angola,1995,24.2,48.055414,9.386791,0.035367,0.84,18.700,22.274928,...,0.0,230991930.0,1140625.0,4.573759e+09,2.258496e+07,1,0,0,13699778.0,3
1,AGO,Angola,1996,24.2,48.055414,9.386791,0.038398,0.84,18.445,22.274928,...,0.0,261331263.9,1348675.0,5.894739e+09,3.042149e+07,1,0,0,14170973.0,3
2,AGO,Angola,1997,24.2,48.055414,9.386791,0.040405,0.84,18.184,22.274928,...,0.0,270465000.0,1423500.0,5.405294e+09,2.844892e+07,1,0,0,14660413.0,3
3,AGO,Angola,1998,24.2,48.055414,9.386791,0.040825,0.84,18.925,22.274928,...,0.0,266760000.0,1470950.0,3.392030e+09,1.870410e+07,1,0,0,15159370.0,3
4,AGO,Angola,1999,24.2,48.055414,9.386791,0.041406,0.84,18.518,22.274928,...,374707057.4,271947000.0,1843250.0,4.689445e+09,3.178494e+07,1,0,0,15667235.0,3


In [46]:
import plotly.express as px
import plotly.graph_objects as go

# ============ CONFIGURATION ============
# Adjust these column names if they differ in your data
COL_COUNTRY_CODE = 'Country Code'
COL_COUNTRY_NAME = 'Country Name'
COL_YEAR = 'Year'
COL_ECI = 'Economic Complexity Index'
COL_GDP_PC = 'GDP per capita (constant prices, PPP)'
COL_PRODUCTION = 'Total_Production_Value'
COL_CLUSTER = 'Cluster'

# ============ MAIN FUNCTION ============
def create_rosling_chart(df):
    """
    Create a Hans Rosling-style animated bubble chart.
    """
    
    # Create a copy to avoid modifying original
    data = df.copy()
    
    # Calculate log GDP per capita
    data['Log GDP per capita'] = np.log(data[COL_GDP_PC])
    
    # Fix cluster colors to 1995 values for each country
    cluster_1995 = data[data[COL_YEAR] == 1995][[COL_COUNTRY_CODE, COL_CLUSTER]].copy()
    cluster_1995 = cluster_1995.rename(columns={COL_CLUSTER: 'Cluster_1995'})
    
    # Merge 1995 clusters back to all years
    data = data.merge(cluster_1995, on=COL_COUNTRY_CODE, how='left')
    
    # Convert cluster to string for discrete colors
    data['Cluster_1995'] = data['Cluster_1995'].astype(int).astype(str)
    
    # Define cluster labels (customize as needed)
    cluster_labels = {
        '0': 'Cluster 0',
        '1': 'Cluster 1',
        '2': 'Cluster 2',
        '3': 'Cluster 3',
        '4': 'Cluster 4'
    }
    data['Cluster_Label'] = data['Cluster_1995'].map(cluster_labels)
    
    # Sort by year
    data = data.sort_values([COL_YEAR, COL_COUNTRY_CODE])
    
    # Define color palette for clusters
    color_map = {
        '0': '#2ecc71',  # Green
        '1': '#e74c3c',  # Red
        '2': '#f39c12',  # Orange
        '3': '#9b59b6',  # Purple
        '4': '#3498db'   # Blue
    }
    
    # Create the animated scatter plot
    fig = px.scatter(
        data,
        x='Log GDP per capita',
        y=COL_ECI,
        size=COL_PRODUCTION,
        color='Cluster_1995',
        color_discrete_map=color_map,
        hover_name=COL_COUNTRY_NAME,
        hover_data={
            COL_COUNTRY_CODE: True,
            'Log GDP per capita': ':.2f',
            COL_ECI: ':.2f',
            COL_PRODUCTION: ':,.0f',
            'Cluster_1995': False,
            COL_YEAR: True
        },
        animation_frame=COL_YEAR,
        animation_group=COL_COUNTRY_CODE,
        size_max=60,
        opacity=0.7,
        labels={
            'Log GDP per capita': 'Log GDP per capita (PPP)',
            COL_ECI: 'Economic Complexity Index',
            'Cluster_1995': 'Cluster (1995)',
            COL_PRODUCTION: 'Production Value (USD)'
        },
        title='Evolution of Economic Complexity vs Income (Sized by Production Value)'
    )
    
    # Update layout
    fig.update_layout(
        xaxis=dict(
            range=[6, 12],
            title='Log GDP per capita (PPP)',
            gridcolor='lightgray',
            showgrid=True
        ),
        yaxis=dict(
            range=[-3, 3],
            title='Economic Complexity Index',
            gridcolor='lightgray',
            showgrid=True
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(size=12),
        legend=dict(
            title='Cluster (1995)',
            yanchor='top',
            y=0.99,
            xanchor='left',
            x=1.02
        ),
        width=1000,
        height=600,
        updatemenus=[
            dict(
                type='buttons',
                showactive=False,
                y=0,
                x=0.1,
                xanchor='right',
                yanchor='top',
                buttons=[
                    dict(
                        label='▶ Play',
                        method='animate',
                        args=[
                            None,
                            dict(
                                frame=dict(duration=500, redraw=True),
                                fromcurrent=True,
                                transition=dict(duration=300, easing='quadratic-in-out')
                            )
                        ]
                    ),
                    dict(
                        label='⏸ Pause',
                        method='animate',
                        args=[
                            [None],
                            dict(
                                frame=dict(duration=0, redraw=False),
                                mode='immediate',
                                transition=dict(duration=0)
                            )
                        ]
                    )
                ]
            )
        ],
        sliders=[{
            'active': 0,
            'yanchor': 'top',
            'xanchor': 'left',
            'currentvalue': {
                'font': {'size': 16},
                'prefix': 'Year: ',
                'visible': True,
                'xanchor': 'center'
            },
            'transition': {'duration': 300, 'easing': 'cubic-in-out'},
            'pad': {'b': 10, 't': 50},
            'len': 0.9,
            'x': 0.05,
            'y': 0,
        }]
    )
    
    # Add border to bubbles
    fig.update_traces(
        marker=dict(
            line=dict(width=1, color='white')
        )
    )
    
    return fig


# ============ RUN WITH YOUR DATA ============
fig = create_rosling_chart(master)
fig.show()

# To save as interactive HTML file:
# fig.write_html('rosling_chart.html')

# To save as interactive HTML file:
# fig.write_html('rosling_chart.html')